<a href="https://colab.research.google.com/github/Metropoliya/AI/blob/claude/content-factory-setup-i3hoa7/content_factory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏭 Контент-завод → Telegram

Публикует пакет постов (видео-рилсы + текст) в твой Telegram-канал.

**Что нужно один раз настроить:**
1. Создай бота у [@BotFather](https://t.me/BotFather): команда `/newbot` → получишь токен вида `1234567890:AA...`
2. Создай/возьми канал и **добавь бота администратором** (право «Публикация сообщений»).
3. `CHAT_ID` — это `@юзернейм_канала` (для публичного) или числовой `-100...` (для приватного).

Дальше просто вставь свои посты в ячейку **НАСТРОЙКИ** и запусти всё сверху вниз (▶︎).


In [ ]:
#@title Шаг 1. Секреты и настройки { display-mode: "form" }
# Вставь свои значения. Токен лучше не хранить в публичном репозитории!
BOT_TOKEN = ""  #@param {type:"string"}
CHAT_ID   = ""  #@param {type:"string"}
DELAY_SEC = 3   #@param {type:"integer"}

assert BOT_TOKEN and CHAT_ID, "Заполни BOT_TOKEN и CHAT_ID выше!"
print("Настройки приняты. Канал:", CHAT_ID)


In [ ]:
#@title Шаг 2. Пакет постов
# Каждый пост: type = "video" | "photo" | "text"
# url     — прямая ссылка на медиа (Telegram скачает сам; например CDN-ссылка Higgsfield)
# caption — текст поста (HTML: <b>жирный</b>, эмодзи и #хэштеги — можно)
POSTS = [
    {
        "type": "video",
        "url": "https://d8j0ntlcm91z4.cloudfront.net/user_3GIv817RpRKyIq1yu4U0ocPsNRp/hf_20260711_062848_c5b9a30a-3ce5-49f8-b6fb-41935d21825f.mp4",
        "caption": (
            "\U0001F3AF <b>\u041f\u0435\u0440\u0435\u0441\u0442\u0430\u043d\u044c \u0436\u0434\u0430\u0442\u044c \u043f\u043e\u043d\u0435\u0434\u0435\u043b\u044c\u043d\u0438\u043a\u0430</b>\n\n"
            "\u041c\u044b \u0432\u0441\u0435 \u0436\u0434\u0451\u043c \u00ab\u043f\u043e\u0434\u0445\u043e\u0434\u044f\u0449\u0435\u0433\u043e \u043c\u043e\u043c\u0435\u043d\u0442\u0430\u00bb. \u041d\u043e \u0435\u0433\u043e \u043d\u0435 \u0441\u0443\u0449\u0435\u0441\u0442\u0432\u0443\u0435\u0442 \u2014 \u0435\u0441\u0442\u044c \u0442\u043e\u043b\u044c\u043a\u043e \u0440\u0435\u0448\u0435\u043d\u0438\u0435, \u043a\u043e\u0442\u043e\u0440\u043e\u0435 \u0442\u044b \u043f\u0440\u0438\u043d\u0438\u043c\u0430\u0435\u0448\u044c \u043f\u0440\u044f\u043c\u043e \u0441\u0435\u0439\u0447\u0430\u0441.\n\n"
            "\U0001F90D \u0421\u043e\u0445\u0440\u0430\u043d\u0438, \u0435\u0441\u043b\u0438 \u043f\u043e\u0440\u0430 \u043d\u0430\u0447\u0438\u043d\u0430\u0442\u044c\n\n"
            "#\u043c\u043e\u0442\u0438\u0432\u0430\u0446\u0438\u044f #\u0441\u0430\u043c\u043e\u0440\u0430\u0437\u0432\u0438\u0442\u0438\u0435 #\u043f\u0441\u0438\u0445\u043e\u043b\u043e\u0433\u0438\u044f"
        ),
    },
]
print(f"\u041f\u043e\u0441\u0442\u043e\u0432 \u0432 \u043f\u0430\u043a\u0435\u0442\u0435: {len(POSTS)}")


In [ ]:
#@title Шаг 3. Опубликовать в Telegram
import time, requests

API = "https://api.telegram.org"

def _send(method, params):
    r = requests.post(f"{API}/bot{BOT_TOKEN}/{method}", data=params, timeout=120)
    return r.json()

def publish(post):
    t = post.get("type", "text")
    cap = post.get("caption", "")
    url = post.get("url", "")
    if t == "video":
        return _send("sendVideo", {"chat_id": CHAT_ID, "video": url, "caption": cap[:1024], "parse_mode": "HTML", "supports_streaming": True})
    if t == "photo":
        return _send("sendPhoto", {"chat_id": CHAT_ID, "photo": url, "caption": cap[:1024], "parse_mode": "HTML"})
    if t == "text":
        return _send("sendMessage", {"chat_id": CHAT_ID, "text": cap[:4096], "parse_mode": "HTML", "disable_web_page_preview": True})
    raise ValueError(f"\u043d\u0435\u0438\u0437\u0432\u0435\u0441\u0442\u043d\u044b\u0439 \u0442\u0438\u043f: {t}")

ok = 0
for i, p in enumerate(POSTS, 1):
    print(f"\u2192 [{i}/{len(POSTS)}] {p.get('type','text')} ...", end=" ")
    res = publish(p)
    if res.get("ok"):
        ok += 1; print("\u2705")
    else:
        print("\u26a0\ufe0f", res)
    if i < len(POSTS):
        time.sleep(DELAY_SEC)

print(f"\n\u0413\u043e\u0442\u043e\u0432\u043e: {ok}/{len(POSTS)} \u043e\u043f\u0443\u0431\u043b\u0438\u043a\u043e\u0432\u0430\u043d\u043e.")
